# NB02 — Metal × CWM Associations (L0–L6 FWL, Bidirectional)

**Forward:** for each metal (As, Cd, Cr, Cu, Ni, Pb, Zn) × causal level (L0–L6), run vectorized FWL across all 6,557 KOs → BH-FDR.

**Reverse:** Ridge regression CWM → metal with spatial block CV → R², SHAP.

**Metal sources:** USGS NGS soil (USA, ~55 K pts), GEMAS (EUR, 4,343 pts), NGSA (AUS, 1,315 pts). Matched to thinned MA samples by KDTree ≤ 50 km. Region dummy included as control.

**Causal levels:**
| Level | Covariates |
|---|---|
| L0 | — |
| L1 | pH (natural spline, 3 df) |
| L2 | +clay, SOC, bulk density, GLiM lithology (dummies) |
| L3 | +log(nearest mine km from mindat) |
| L4 | +MAT, MAP, temp_seasonality, precip_seasonality |
| L5 | +Shannon diversity, phylum RA (top 8) |
| L6 | +elevation, NDVI, ESA landcover dummies |

In [1]:

import sys, os
sys.path.insert(0, '/home/hmacgregor/BERIL-research-observatory/tools')
from figure_style import apply_style, save, PALETTE, METAL_COLORS, FIGW, ROW_H, grid_h, annotate_n
apply_style()

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.spatial import KDTree
from scipy.stats import norm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

PROJECT = Path('/home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_cwm')
DATA    = PROJECT / 'data'
FIGS    = PROJECT / 'figures'

try:
    from berdl_notebook_utils.setup_spark_session import get_spark_session
    spark = get_spark_session()
except Exception as e:
    print(f'berdl_notebook_utils failed ({e}), trying getOrCreate')
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

print(f'Spark {spark.version}')
print(f'DATA: {DATA}')


Spark 4.0.1
DATA: /home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_cwm/data


In [2]:
# Load NB00/NB01 outputs
thinned      = pd.read_parquet(DATA / 'nb00_thinned_samples.parquet')
genus_counts = pd.read_parquet(DATA / 'nb01_genus_counts.parquet')
cwm_long     = pd.read_parquet(DATA / 'nb01_cwm.parquet')

print(f'Thinned samples:     {len(thinned):,}')
print(f'CWM: {cwm_long["sample_id"].nunique():,} samples × {cwm_long["ko_id"].nunique():,} KOs')

# Pivot CWM to wide format (samples × KOs) — used throughout for FWL
print('Pivoting CWM to wide format…')
cwm_wide = cwm_long.pivot_table(index='sample_id', columns='ko_id', values='cwm', fill_value=0.0)
cwm_wide = cwm_wide.astype(np.float32)
print(f'CWM wide: {cwm_wide.shape}')

# Composite pH: measured > OLM÷10 > SoilGrids raster (covers 966/1006 otherwise-missing)
ph_raw = pd.to_numeric(thinned['ph'], errors='coerce')
ph_olm = pd.to_numeric(thinned['olm_soil_ph_0cm_H2O'], errors='coerce') / 10.0
thinned['ph_final'] = ph_raw.where(ph_raw.notna(), ph_olm.where(ph_olm.notna(), thinned['ph_soilgrids']))
thinned['ph_is_modelled'] = ph_raw.isna().astype(np.float32)
print(f'pH available: {thinned["ph_final"].notna().sum():,} / {len(thinned):,}')

# Register temp view for Spark joins
ids_spark = spark.createDataFrame(thinned[['sample_id']])
ids_spark.createOrReplaceTempView('thinned_sample_ids')
print('Registered thinned_sample_ids temp view')

Thinned samples:     4,884


CWM: 4,868 samples × 6,557 KOs
Pivoting CWM to wide format…


CWM wide: (4868, 6557)
pH available: 4,844 / 4,884
Registered thinned_sample_ids temp view


In [3]:

# Fetch soil properties: OLM (from sample_metadata) with SoilGrids fallback for missing values.

if (DATA / 'nb02_soil_props.parquet').exists():
    print('Loading soil_props from cache')
    soil_props = pd.read_parquet(DATA / 'nb02_soil_props.parquet')
else:
    # ---- Step 1: OLM columns pre-joined in sample_metadata ----
    result = spark.sql("""
        SELECT m.sample_id,
               m.olm_soil_clay_0cm_pct           AS clay_pct,
               m.olm_soil_organic_matter_0cm_pct  AS som_pct,
               m.olm_soil_bulk_density_0cm_g_cm3  AS bulk_density,
               m.soil_moisture_root_cm3_cm3        AS soil_moisture
        FROM arkinlab.microbeatlas.sample_metadata m
        JOIN thinned_sample_ids ts ON m.sample_id = ts.sample_id
    """)
    result.attrs = {}
    soil_props = result.toPandas()
    soil_props.attrs = {}
    # Cast to float64 immediately — avoids float32/float64 dtype conflict on fill
    for c in ['clay_pct','som_pct','bulk_density','soil_moisture']:
        soil_props[c] = pd.to_numeric(soil_props[c], errors='coerce').astype(np.float64)

    print('OLM coverage:')
    for c in ['clay_pct','som_pct','bulk_density','soil_moisture']:
        n = soil_props[c].notna().sum()
        print(f'  {c}: {n:,}/{len(soil_props):,}')

    # ---- Step 2: SoilGrids fallback via soilgrids_master ----
    print('\nFetching soilgrids_master schema…')
    sg_schema = spark.sql('SELECT * FROM arkinlab.envdbs.soilgrids_master LIMIT 3').toPandas()
    print('soilgrids_master columns:', list(sg_schema.columns))
    sg_cols = list(sg_schema.columns)

    clay_col = next((c for c in sg_cols if 'clay' in c.lower()), None)
    soc_col  = next((c for c in sg_cols if 'organic_carbon' in c.lower() or c.lower().startswith('soc')), None)
    bd_col   = next((c for c in sg_cols if 'bulk_density' in c.lower() or 'bdod' in c.lower()), None)
    lat_col  = next((c for c in sg_cols if c.lower() == 'lat'), 'lat')
    lon_col  = next((c for c in sg_cols if c.lower() == 'lon'), 'lon')
    print(f'Mapped: lat={lat_col}, lon={lon_col}, clay={clay_col}, soc={soc_col}, bd={bd_col}')

    select_cols = [f'`{lat_col}`', f'`{lon_col}`']
    alias_map = {}
    if clay_col: select_cols.append(f'`{clay_col}` AS sg_clay'); alias_map['sg_clay'] = 'clay_pct'
    if soc_col:  select_cols.append(f'`{soc_col}` AS sg_soc');   alias_map['sg_soc']  = 'som_pct'
    if bd_col:   select_cols.append(f'`{bd_col}` AS sg_bd');     alias_map['sg_bd']   = 'bulk_density'

    sg_result = spark.sql(f"""
        SELECT {', '.join(select_cols)}
        FROM arkinlab.envdbs.soilgrids_master
        WHERE `{lat_col}` IS NOT NULL AND `{lon_col}` IS NOT NULL
    """)
    sg_result.attrs = {}
    sg_full = sg_result.toPandas()
    sg_full.attrs = {}
    print(f'soilgrids_master rows: {len(sg_full):,}')

    for col in ['sg_clay', 'sg_soc', 'sg_bd']:
        if col in sg_full.columns:
            sg_full[col] = pd.to_numeric(sg_full[col], errors='coerce').astype(np.float64)

    # Unit auto-correction based on value ranges
    if 'sg_clay' in sg_full.columns:
        med = sg_full['sg_clay'].median()
        if med > 20:  # stored as g/kg instead of %
            sg_full['sg_clay'] /= 10.0
            print(f'Clay: converted g/kg→% (median was {med:.1f})')
    if 'sg_soc' in sg_full.columns:
        med = sg_full['sg_soc'].median()
        if med > 100:  # stored as dg/kg instead of g/kg
            sg_full['sg_soc'] /= 10.0
            print(f'SOC: converted dg/kg→g/kg (median was {med:.1f})')
    if 'sg_bd' in sg_full.columns:
        med = sg_full['sg_bd'].median()
        if med > 5:  # stored as cg/cm³ instead of g/cm³
            sg_full['sg_bd'] /= 100.0
            print(f'BD: converted cg/cm³→g/cm³ (median was {med:.1f})')

    # KDTree join: soilgrids → thinned samples
    tree_sg = KDTree(sg_full[[lat_col, lon_col]].values)
    _, idxs_sg = tree_sg.query(thinned[['lat','lon']].values, k=1)

    # Merge: OLM is primary; soilgrids fills gaps via combine_first on indexed frames
    soil_indexed = soil_props.set_index('sample_id')
    sample_ids = thinned['sample_id'].values

    if 'sg_clay' in sg_full.columns:
        sg_s = pd.Series(sg_full['sg_clay'].values[idxs_sg], index=sample_ids, dtype=np.float64)
        soil_indexed['clay_pct'] = soil_indexed['clay_pct'].combine_first(sg_s)
    if 'sg_soc' in sg_full.columns:
        sg_s = pd.Series(sg_full['sg_soc'].values[idxs_sg], index=sample_ids, dtype=np.float64)
        soil_indexed['som_pct'] = soil_indexed['som_pct'].combine_first(sg_s)
    if 'sg_bd' in sg_full.columns:
        sg_s = pd.Series(sg_full['sg_bd'].values[idxs_sg], index=sample_ids, dtype=np.float64)
        soil_indexed['bulk_density'] = soil_indexed['bulk_density'].combine_first(sg_s)

    soil_props = soil_indexed.reset_index()
    soil_props.attrs = {}
    soil_props.to_parquet(DATA / 'nb02_soil_props.parquet', index=False)

    print('\nCoverage after SoilGrids fill:')
    for c in ['clay_pct','som_pct','bulk_density','soil_moisture']:
        n = soil_props[c].notna().sum()
        print(f'  {c}: {n:,}/{len(soil_props):,}')

print('Soil props shape:', soil_props.shape)


Loading soil_props from cache
Soil props shape: (4884, 5)


In [4]:

# Fetch GLiM lithology (full table, KDTree join to thinned samples).
if (DATA / 'nb02_lithology.parquet').exists():
    print('Loading lithology from cache')
    lith = pd.read_parquet(DATA / 'nb02_lithology.parquet')
else:
    result = spark.sql("""
        SELECT lat, lon, lithology_class AS lith_class
        FROM arkinlab.envdbs.global_lithology_glim
        WHERE lat IS NOT NULL AND lon IS NOT NULL
    """)
    result.attrs = {}
    lith_full = result.toPandas()
    lith_full.attrs = {}
    print(f'GLiM rows: {len(lith_full):,}')

    tree_lith = KDTree(lith_full[['lat','lon']].values)
    dists, idxs = tree_lith.query(thinned[['lat','lon']].values, k=1)
    lith = pd.DataFrame({
        'sample_id': thinned['sample_id'].values,
        'lith_class': lith_full['lith_class'].values[idxs],
        'lith_dist_deg': dists,
    })
    lith.to_parquet(DATA / 'nb02_lithology.parquet', index=False)
    print(f'Assigned lithology to {len(lith):,} samples')

print('Lithology classes:')
print(lith['lith_class'].value_counts())

lith_dummies = pd.get_dummies(lith.set_index('sample_id')['lith_class'], prefix='lith', drop_first=True).astype(np.float32)


Loading lithology from cache
Lithology classes:
lith_class
Unconsolidated Sediment (SU)      1299
Pyroclastics (PY)                  901
Carbonate Sedimentary (SC)         706
Evaporites (EV)                    494
Intermediate Volcanic (VI)         429
Basic Volcanic (VB)                365
Siliciclastic Sedimentary (SS)     176
Acid Volcanic (VA)                  92
Acid Plutonic (PA)                  76
Basic Plutonic (PB)                 45
Intermediate Plutonic (PI)          42
Metamorphic (MT)                    33
Mixed Sedimentary (SM)              29
Name: count, dtype: int64


In [5]:
# Fetch WorldClim bio_4 (temp seasonality) and bio_15 (precip seasonality).
# lat/lon in worldclim_master are STRING-typed — use TRY_CAST throughout.
if (DATA / 'nb02_worldclim_seasonal.parquet').exists():
    print('Loading worldclim_seasonal from cache')
    wc = pd.read_parquet(DATA / 'nb02_worldclim_seasonal.parquet')
else:
    result = spark.sql("""
        SELECT TRY_CAST(lat AS DOUBLE) AS lat,
               TRY_CAST(lon AS DOUBLE) AS lon,
               TRY_CAST(bio_4  AS DOUBLE) AS bio_4,
               TRY_CAST(bio_15 AS DOUBLE) AS bio_15
        FROM arkinlab.envdbs.worldclim_master
        WHERE TRY_CAST(lat AS DOUBLE) IS NOT NULL
          AND TRY_CAST(lon AS DOUBLE) IS NOT NULL
    """)
    result.attrs = {}
    wc_full = result.toPandas()
    wc_full.attrs = {}
    print(f'worldclim_master rows: {len(wc_full):,}')

    tree_wc = KDTree(wc_full[['lat','lon']].values)
    dists, idxs = tree_wc.query(thinned[['lat','lon']].values, k=1)
    wc = pd.DataFrame({
        'sample_id': thinned['sample_id'].values,
        'temp_seasonality': wc_full['bio_4'].values[idxs],
        'precip_seasonality': wc_full['bio_15'].values[idxs],
    })
    wc.to_parquet(DATA / 'nb02_worldclim_seasonal.parquet', index=False)
    print(f'Fetched WorldClim seasonality: {wc.shape}')

print('WorldClim seasonal coverage:')
for c in ['temp_seasonality','precip_seasonality']:
    print(f'  {c}: {wc[c].notna().sum():,}/{len(wc):,}')


Loading worldclim_seasonal from cache
WorldClim seasonal coverage:
  temp_seasonality: 4,883/4,884
  precip_seasonality: 4,883/4,884


In [6]:
# Fetch GEMAS (European Agricultural and Grazing land Soil metal survey).
if (DATA / 'nb02_gemas.parquet').exists():
    print('Loading GEMAS from cache')
    gemas = pd.read_parquet(DATA / 'nb02_gemas.parquet')
else:
    spark.sql('SELECT * FROM arkinlab.envdbs.gemas LIMIT 3').show(truncate=False, vertical=True)
    result = spark.sql('SELECT * FROM arkinlab.envdbs.gemas')
    result.attrs = {}
    gemas = result.toPandas()
    gemas.attrs = {}
    gemas.to_parquet(DATA / 'nb02_gemas.parquet', index=False)
    print(f'GEMAS shape: {gemas.shape}')
    print('Columns:', list(gemas.columns))

print(f'GEMAS rows: {len(gemas):,}')
print('Columns:', list(gemas.columns))

Loading GEMAS from cache
GEMAS rows: 4,343
Columns: ['id', 'country', 'country_id', 'type', 'type2', 'longitude', 'latitude', 'x_laea', 'y_laea', 'altitude', 'aps_1960_1990', 'amt_1960_1990', 'aps_1970_2000', 'amt_1970_2000', 'climate', 'soiltype', 'soilclass', 'cgsg', 'litho', 'pm_hart', 'pm_guen', 'ecoregio', 'pd_2005', 'pd_2020', 'ag_ppm_ar', 'al_ppm_ar', 'as_ppm_ar', 'au_ppm_ar', 'b_ppm_ar', 'ba_ppm_ar', 'be_ppm_ar', 'bi_ppm_ar', 'ca_ppm_ar', 'cd_ppm_ar', 'ce_ppm_ar', 'co_ppm_ar', 'cr_ppm_ar', 'cs_ppm_ar', 'cu_ppm_ar', 'fe_ppm_ar', 'ga_ppm_ar', 'ge_ppm_ar', 'hf_ppm_ar', 'hg_ppm_ar', 'in_ppm_ar', 'k_ppm_ar', 'la_ppm_ar', 'li_ppm_ar', 'mg_ppm_ar', 'mn_ppm_ar', 'mo_ppm_ar', 'na_ppm_ar', 'nb_ppm_ar', 'ni_ppm_ar', 'p_ppm_ar', 'pb_ppm_ar', 'pd_ppm_ar', 'pt_ppm_ar', 'rb_ppm_ar', 're_ppm_ar', 's_ppm_ar', 'sb_ppm_ar', 'sc_ppm_ar', 'se_ppm_ar', 'sn_ppm_ar', 'sr_ppm_ar', 'ta_ppm_ar', 'te_ppm_ar', 'th_ppm_ar', 'ti_ppm_ar', 'tl_ppm_ar', 'u_ppm_ar', 'v_ppm_ar', 'w_ppm_ar', 'y_ppm_ar', 'zn_ppm_ar

In [7]:
# Fetch NGSA (National Geochemical Survey of Australia).
if (DATA / 'nb02_ngsa.parquet').exists():
    print('Loading NGSA from cache')
    ngsa = pd.read_parquet(DATA / 'nb02_ngsa.parquet')
else:
    spark.sql('SELECT * FROM arkinlab.envdbs.ngsa_geochemistry LIMIT 3').show(truncate=False, vertical=True)
    result = spark.sql('SELECT * FROM arkinlab.envdbs.ngsa_geochemistry')
    result.attrs = {}
    ngsa = result.toPandas()
    ngsa.attrs = {}
    ngsa.to_parquet(DATA / 'nb02_ngsa.parquet', index=False)
    print(f'NGSA shape: {ngsa.shape}')
    print('Columns:', list(ngsa.columns))

print(f'NGSA rows: {len(ngsa):,}')
print('Columns:', list(ngsa.columns))

Loading NGSA from cache
NGSA rows: 1,315
Columns: ['id', 'siteid', 'date_sampled', 'lat', 'lon', 'state', 'duplicate_code', 'duplicate_siteid', 'sampleid', 'grain_size', 'depth', 'ag_icp_ms_mg_kg_0_03', 'ag_ar_mg_kg_0_002', 'ag_mmi_me_mg_kg_0_001', 'al_xrf_mg_kg_26', 'al_ar_mg_kg_100_100k', 'al_mmi_me_mg_kg_1', 'as_icp_ms_mg_kg_0_4', 'as_ar_mg_kg_0_1', 'as_mmi_me_mg_kg_0_01', 'au_fa_mg_kg_0_001', 'au_ar_mg_kg_0_0001', 'au_mmi_me_mg_kg_0_0001', 'b_ar_mg_kg_1', 'ba_icp_ms_mg_kg_0_5', 'ba_ar_mg_kg_0_5', 'ba_mmi_me_mg_kg_0_01', 'be_icp_ms_mg_kg_1_1', 'be_ar_mg_kg_0_1', 'bi_icp_ms_mg_kg_0_02', 'bi_ar_mg_kg_0_02', 'bi_mmi_me_mg_kg_0_001', 'ca_xrf_mg_kg_14', 'ca_ar_mg_kg_100', 'ca_mmi_me_mg_kg_10', 'cd_icp_ms_mg_kg_0_1', 'cd_ar_mg_kg_0_01', 'cd_mmi_me_mg_kg_0_001', 'ce_icp_ms_mg_kg_0_03', 'ce_ar_mg_kg_0_01', 'ce_mmi_me_mg_kg_0_005', 'cl_xrf_mg_kg_10', 'co_icp_ms_mg_kg_0_1', 'co_ar_mg_kg_0_1', 'co_mmi_me_mg_kg_0_005', 'cr_icp_ms_mg_kg_0_5', 'cr_ar_mg_kg_0_5', 'cr_mmi_me_mg_kg_0_001', 'cs_icp_m

In [8]:
# Compute nearest mine distance for each thinned sample from mindat.csv (157K localities).
# Mindat has global coverage; 'elements_inc' lists elements extracted.
MINDAT_PATH = Path('/home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_metal_ecology/data/mindat.csv')

if (DATA / 'nb02_mine_dist.parquet').exists():
    print('Loading mine_dist from cache')
    mine_dist = pd.read_parquet(DATA / 'nb02_mine_dist.parquet')
else:
    md = pd.read_csv(MINDAT_PATH, low_memory=False, usecols=['latitude','longitude','elements_inc'])
    md = md.dropna(subset=['latitude','longitude'])
    md['latitude']  = pd.to_numeric(md['latitude'],  errors='coerce')
    md['longitude'] = pd.to_numeric(md['longitude'], errors='coerce')
    md = md.dropna(subset=['latitude','longitude'])
    md = md[(md['latitude'].between(-90,90)) & (md['longitude'].between(-180,180))]
    print(f'Mindat localities: {len(md):,}')

    tree_mine = KDTree(md[['latitude','longitude']].values)
    dists_deg, idxs = tree_mine.query(thinned[['lat','lon']].values, k=1)
    # Convert degrees to km (approximate, equatorial degrees)
    dists_km = dists_deg * 111.32

    mine_dist = pd.DataFrame({
        'sample_id':       thinned['sample_id'].values,
        'mine_dist_km':    dists_km,
        'mine_elements':   md['elements_inc'].values[idxs],
    })
    mine_dist.to_parquet(DATA / 'nb02_mine_dist.parquet', index=False)
    print(f'Assigned mine distance to {len(mine_dist):,} samples')

print(mine_dist['mine_dist_km'].describe())
# log-transform for models (min 0.1 km to avoid log(0))
mine_dist['log_mine_km'] = np.log10(mine_dist['mine_dist_km'].clip(0.1))

Loading mine_dist from cache
count    4884.000000
mean       80.951068
std       118.228212
min         0.061346
25%        22.932862
50%        49.936719
75%        97.677645
max      3177.831160
Name: mine_dist_km, dtype: float64


In [9]:
# Fetch genus → phylum mapping from otu_metadata for genera in our genus_counts.
# Used to compute phylum RA per sample (L5 covariate).
if (DATA / 'nb02_genus_phylum.parquet').exists():
    print('Loading genus_phylum from cache')
    genus_phylum = pd.read_parquet(DATA / 'nb02_genus_phylum.parquet')
else:
    our_genera = genus_counts['genus_lower'].unique().tolist()
    genera_spark = spark.createDataFrame(
        pd.DataFrame({'genus_lower': our_genera})
    )
    genera_spark.createOrReplaceTempView('our_genera')

    result = spark.sql("""
        SELECT LOWER(om.genus) AS genus_lower,
               LOWER(TRIM(try_element_at(SPLIT(om.tax, ';'), 3))) AS phylum_lower
        FROM arkinlab.microbeatlas.otu_metadata om
        JOIN our_genera og ON LOWER(om.genus) = og.genus_lower
        WHERE om.genus IS NOT NULL AND om.genus != ''
        GROUP BY LOWER(om.genus), LOWER(TRIM(try_element_at(SPLIT(om.tax, ';'), 3)))
    """)
    result.attrs = {}
    genus_phylum = result.toPandas()
    genus_phylum.attrs = {}
    genus_phylum = genus_phylum.dropna(subset=['phylum_lower'])
    genus_phylum = genus_phylum.drop_duplicates(subset='genus_lower', keep='first')
    genus_phylum.to_parquet(DATA / 'nb02_genus_phylum.parquet', index=False)
    print(f'genus_phylum: {len(genus_phylum):,} genera mapped to phyla')

print(f'Genera with phylum: {len(genus_phylum):,}')
print('Top phyla:', genus_phylum['phylum_lower'].value_counts().head(10).to_dict())


Loading genus_phylum from cache


Genera with phylum: 4,057
Top phyla: {'alphaproteobacteria': 509, 'gammaproteobacteria': 432, 'clostridia': 370, 'actinomycetia': 329, 'bacilli': 303, 'betaproteobacteria': 197, 'flavobacteriia': 173, 'deltaproteobacteria': 159, '': 150, 'bacteroidia': 88}


In [10]:
from scipy.stats import entropy as scipy_entropy

# Shannon diversity per sample from genus_counts (genus-level 16S, not OTU-level)
def _shannon(grp):
    counts = grp['genus_count'].values
    counts = counts[counts > 0]
    return scipy_entropy(counts)

shannon_df = genus_counts.groupby('sample_id').apply(_shannon).reset_index()
shannon_df.columns = ['sample_id', 'shannon']
print(f'Shannon computed for {len(shannon_df):,} samples')
print(shannon_df['shannon'].describe())

# Phylum RA per sample: top 8 phyla as L5 covariates
gc_phy = genus_counts.merge(genus_phylum[['genus_lower','phylum_lower']], on='genus_lower', how='left')
gc_phy['phylum_lower'] = gc_phy['phylum_lower'].fillna('unknown')

phy_counts = gc_phy.groupby(['sample_id','phylum_lower'])['genus_count'].sum().reset_index()
sample_totals = genus_counts.groupby('sample_id')['genus_count'].sum().rename('total')
phy_ra = phy_counts.join(sample_totals, on='sample_id')
phy_ra['phy_ra'] = phy_ra['genus_count'] / phy_ra['total']

# Select top 8 phyla by mean RA across all samples
top8_phyla = (
    phy_ra.groupby('phylum_lower')['phy_ra'].mean()
    .sort_values(ascending=False)
    .head(8)
    .index.tolist()
)
print('Top 8 phyla:', top8_phyla)

phy_wide = (
    phy_ra[phy_ra['phylum_lower'].isin(top8_phyla)]
    .pivot_table(index='sample_id', columns='phylum_lower', values='phy_ra', fill_value=0.0)
    .astype(np.float32)
)
print(f'Phylum RA matrix: {phy_wide.shape}')

Shannon computed for 4,879 samples
count    4879.000000
mean        3.091445
std         1.227535
min         0.000000
25%         2.167280
50%         3.362157
75%         4.102803
max         5.454900
Name: shannon, dtype: float64


Top 8 phyla: ['alphaproteobacteria', 'sordariomycetes', 'actinomycetia', 'gammaproteobacteria', 'bryopsida', 'insecta', 'nitrososphaeria', 'wallemiomycetes']
Phylum RA matrix: (4820, 8)


In [11]:
# Build USGS soil metal wide-format and join to thinned MA samples by KDTree ≤ 50 km.
USGS_PATH   = Path('/home/hmacgregor/data/envdbs/usgs_geochem/usgs_geochem.parquet')
JOINED_PATH = Path('/home/hmacgregor/data/envdbs/usgs_geochem/usgs_geochem_joined.parquet')

if (DATA / 'nb02_usgs_metal_joined.parquet').exists():
    print('Loading usgs_metal_joined from cache')
    usgs_join = pd.read_parquet(DATA / 'nb02_usgs_metal_joined.parquet')
else:
    u_meta  = pd.read_parquet(USGS_PATH)
    u_long  = pd.read_parquet(JOINED_PATH)

    # Soil samples only (primary_class == 'soil')
    soil_ids = set(u_meta[u_meta['primary_class'] == 'soil']['lab_id'].dropna())

    # Pivot primary 7 metals + extended elements (all with ≥500 soil measurements)
    primary_metals = ['As','Cd','Cr','Cu','Ni','Pb','Zn']
    primary_params = [f'{m}_ppm_ES_SQ' for m in primary_metals]
    # Extended: any _ppm_ES_SQ parameter with ≥500 soil samples
    param_counts = u_long[u_long['lab_id'].isin(soil_ids)]['parameter'].value_counts()
    extended_params = [p for p in param_counts[param_counts >= 500].index if p.endswith('_ppm_ES_SQ')]
    all_params = list(set(primary_params) | set(extended_params))

    metal_long = u_long[
        u_long['parameter'].isin(all_params) & u_long['lab_id'].isin(soil_ids)
    ].copy()
    metal_wide = metal_long.pivot_table(
        index='lab_id', columns='parameter', values='qualified_value', aggfunc='first'
    ).reset_index()
    metal_wide.columns = [
        c.replace('_ppm_ES_SQ', '') if c != 'lab_id' else c
        for c in metal_wide.columns
    ]

    coords = u_meta[u_meta['primary_class'] == 'soil'][['lab_id','latitude','longitude']].dropna()
    usgs_soil = coords.merge(metal_wide, on='lab_id', how='inner')
    print(f'USGS soil points: {len(usgs_soil):,}')

    # KDTree join to thinned MA samples (≤ 50 km ≈ 0.45°)
    tree_usgs = KDTree(usgs_soil[['latitude','longitude']].values)
    dists_deg, idxs = tree_usgs.query(thinned[['lat','lon']].values, k=1)
    dists_km = dists_deg * 111.32
    mask = dists_km <= 50

    metal_cols = [c for c in usgs_soil.columns if c not in ['lab_id','latitude','longitude']]
    usgs_join = pd.DataFrame({'sample_id': thinned['sample_id'].values})
    for col in metal_cols:
        usgs_join[col] = np.where(mask, usgs_soil[col].values[idxs], np.nan)
    usgs_join['usgs_dist_km'] = np.where(mask, dists_km, np.nan)
    usgs_join['region'] = np.where(mask, 'USA', pd.NA)

    usgs_join.to_parquet(DATA / 'nb02_usgs_metal_joined.parquet', index=False)
    print(f'MA samples with USGS metals ≤50 km: {mask.sum():,}')

n_usgs = usgs_join['region'].notna().sum()
print(f'MA samples with USGS metals: {n_usgs:,}')
metal_cols_usgs = [c for c in usgs_join.columns if c not in ['sample_id','usgs_dist_km','region']]
print(f'Metal/element columns: {len(metal_cols_usgs)}')

Loading usgs_metal_joined from cache
MA samples with USGS metals: 545
Metal/element columns: 56


In [12]:
# Combine USGS (USA, all elements), GEMAS (EUR), NGSA (AUS) metal measurements.
# GEMAS/NGSA column names hardcoded from prior schema inspection.

GEMAS_MAP = {  # canonical -> gemas aqua-regia column
    'As': 'as_ppm_ar', 'Cd': 'cd_ppm_ar', 'Cr': 'cr_ppm_ar',
    'Cu': 'cu_ppm_ar', 'Ni': 'ni_ppm_ar', 'Pb': 'pb_ppm_ar', 'Zn': 'zn_ppm_ar',
}
NGSA_MAP = {   # canonical -> ngsa ICP-MS column
    'As': 'as_icp_ms_mg_kg_0_4', 'Cd': 'cd_icp_ms_mg_kg_0_1',
    'Cr': 'cr_icp_ms_mg_kg_0_5', 'Cu': 'cu_icp_ms_mg_kg_0_2',
    'Ni': 'ni_icp_ms_mg_kg_0_5', 'Pb': 'pb_icp_ms_mg_kg_0_1',
    'Zn': 'zn_icp_ms_mg_kg_0_9',
}
PRIMARY_METALS = list(GEMAS_MAP.keys())

usgs_element_cols = [c for c in usgs_join.columns
                     if c not in ('sample_id', 'usgs_dist_km', 'region')]
print(f'USGS element columns: {len(usgs_element_cols)}')
print(sorted(usgs_element_cols))

# GEMAS join (EUR, <=50 km)
gemas_lat = pd.to_numeric(gemas['latitude'], errors='coerce')
gemas_lon = pd.to_numeric(gemas['longitude'], errors='coerce')
gemas_ok = gemas_lat.notna() & gemas_lon.notna()
tree_gemas = KDTree(np.column_stack([gemas_lat[gemas_ok].values,
                                      gemas_lon[gemas_ok].values]))
dists_g, idxs_g = tree_gemas.query(thinned[['lat','lon']].values, k=1)
mask_g = dists_g * 111.32 <= 50
gemas_rows = np.where(gemas_ok)[0]

gemas_join = pd.DataFrame({'sample_id': thinned['sample_id'].values})
for canonical, col in GEMAS_MAP.items():
    raw = pd.to_numeric(gemas[col].iloc[gemas_rows].values[idxs_g], errors='coerce')
    gemas_join[canonical] = np.where(mask_g, raw, np.nan)
gemas_join['region'] = np.where(mask_g, 'EUR', pd.NA)
print(f'GEMAS <=50 km: {mask_g.sum():,}')

# NGSA join (AUS, <=50 km)
ngsa_lat = pd.to_numeric(ngsa['lat'], errors='coerce')
ngsa_lon = pd.to_numeric(ngsa['lon'], errors='coerce')
ngsa_ok = ngsa_lat.notna() & ngsa_lon.notna()
tree_ngsa = KDTree(np.column_stack([ngsa_lat[ngsa_ok].values,
                                     ngsa_lon[ngsa_ok].values]))
dists_n, idxs_n = tree_ngsa.query(thinned[['lat','lon']].values, k=1)
mask_n = dists_n * 111.32 <= 50
ngsa_rows = np.where(ngsa_ok)[0]

ngsa_join = pd.DataFrame({'sample_id': thinned['sample_id'].values})
for canonical, col in NGSA_MAP.items():
    raw = pd.to_numeric(ngsa[col].iloc[ngsa_rows].values[idxs_n], errors='coerce')
    ngsa_join[canonical] = np.where(mask_n, raw, np.nan)
ngsa_join['region'] = np.where(mask_n, 'AUS', pd.NA)
print(f'NGSA <=50 km: {mask_n.sum():,}')

# Build combined: all USGS elements + EUR/AUS fill for primary 7
combined = pd.DataFrame({'sample_id': thinned['sample_id'].values})
for col in usgs_element_cols:
    combined[col] = usgs_join[col].values

for m in PRIMARY_METALS:
    if m in combined.columns:
        base_s = pd.Series(combined[m].values, dtype=np.float64)
        eur_s  = pd.Series(gemas_join[m].values, dtype=np.float64)
        aus_s  = pd.Series(ngsa_join[m].values, dtype=np.float64)
        combined[m] = base_s.fillna(eur_s).fillna(aus_s).values

region_u = usgs_join['region'].fillna('')
region_g = gemas_join['region'].fillna('')
region_n = ngsa_join['region'].fillna('')
combined['region'] = np.where(region_u=='USA', 'USA',
                     np.where(region_g=='EUR', 'EUR',
                     np.where(region_n=='AUS', 'AUS', pd.NA)))
combined.attrs = {}
combined.to_parquet(DATA / 'nb02_combined_metals.parquet', index=False)

# Coverage report
print('\nRegion counts:', combined['region'].value_counts().to_dict())
print('\nElement coverage (n_notna / n_positive):')
metal_coverage = {}
for col in usgs_element_cols:
    arr = pd.to_numeric(combined[col], errors='coerce')
    n_nn  = int(arr.notna().sum())
    n_pos = int((arr > 0).sum())
    metal_coverage[col] = {'n_notna': n_nn, 'n_positive': n_pos}
    if n_nn >= 30:
        print(f'  {col:12s}: {n_nn:5,} non-NaN, {n_pos:5,} positive')

METAL_LIST = sorted([c for c in usgs_element_cols
                     if metal_coverage[c]['n_positive'] >= 30])
print(f'\nMETAL_LIST: {len(METAL_LIST)} elements with >=30 positive values: {METAL_LIST}')


USGS element columns: 56
['Ag', 'As', 'Au', 'B', 'Ba', 'Be', 'Bi', 'Cd', 'Ce', 'Co', 'Cr', 'Cu', 'Dy', 'Er', 'Eu', 'Ga', 'Gd', 'Ge', 'Hf', 'Hg', 'Ho', 'In', 'Ir', 'La', 'Li', 'Lu', 'Mo', 'Nb', 'Nd', 'Ni', 'Os', 'Pb', 'Pd', 'Pr', 'Pt', 'Re', 'Rh', 'Ru', 'Sb', 'Sc', 'Sm', 'Sn', 'Sr', 'Ta', 'Tb', 'Te', 'Th', 'Tl', 'Tm', 'U', 'V', 'W', 'Y', 'Yb', 'Zn', 'Zr']
GEMAS <=50 km: 921
NGSA <=50 km: 236

Region counts: {'EUR': 921, 'USA': 545, 'AUS': 236}

Element coverage (n_notna / n_positive):
  Ag          :   540 non-NaN,    12 positive
  As          : 1,666 non-NaN, 1,143 positive
  Au          :   531 non-NaN,     0 positive
  B           :   534 non-NaN,   380 positive
  Ba          :   538 non-NaN,   533 positive
  Be          :   538 non-NaN,   221 positive
  Bi          :   534 non-NaN,     2 positive
  Cd          : 1,531 non-NaN,   998 positive
  Ce          :   439 non-NaN,    45 positive
  Co          :   538 non-NaN,   436 positive
  Cr          : 1,698 non-NaN, 1,693 positive
  Cu 

In [13]:
# Figure: sample count per primary metal, stacked by region
region_order = ['USA', 'EUR', 'AUS']
region_colors = dict(zip(region_order, PALETTE[:3]))

fig, ax = plt.subplots(figsize=(FIGW['1.5col'], ROW_H * 0.8))

bottoms = np.zeros(len(PRIMARY_METALS))
for region in region_order:
    counts = []
    for m in PRIMARY_METALS:
        n = combined[(combined['region'] == region) & combined[m].notna()][m].shape[0]
        counts.append(n)
    ax.bar(PRIMARY_METALS, counts, bottom=bottoms, color=region_colors[region],
           edgecolor='k', linewidth=0.5, label=region)
    bottoms += np.array(counts, dtype=float)

ax.set_xlabel('Metal')
ax.set_ylabel('MicrobeAtlas samples matched')
ax.set_title('Metal measurement coverage (≤ 50 km radius)', fontsize=10)
ax.legend(fontsize=8, frameon=False)

save(fig, FIGS / 'fig_nb02_metal_coverage')

In [14]:

# Assemble per-sample covariate matrix for L0–L6.
# clay_pct, som_pct, bulk_density are filled from OLM+SoilGrids (near-complete coverage).
# soil_moisture has no SoilGrids fallback → impute_median only for that column.

base = thinned[['sample_id','lat','lon','ph_final','ph_is_modelled',
                 'dem_elevation_m','era5_mean_2m_air_temp_k',
                 'era5_total_precipitation_mm','ndvi']].copy()
base = base.set_index('sample_id')

# Join soil properties (OLM+SoilGrids filled)
base = base.join(soil_props.set_index('sample_id')[['clay_pct','som_pct','bulk_density','soil_moisture']], how='left')

# Remaining joins
base = base.join(lith_dummies, how='left')
base = base.join(wc.set_index('sample_id'), how='left')
base = base.join(mine_dist[['sample_id','log_mine_km']].set_index('sample_id'), how='left')
base = base.join(shannon_df.set_index('sample_id'), how='left')
base = base.join(phy_wide, how='left')
region_dummies = pd.get_dummies(combined.set_index('sample_id')['region'],
                                 prefix='region', drop_first=True).astype(np.float32)
base = base.join(region_dummies, how='left')

# pH cubic polynomial basis (centered, 3 df)
ph_arr = base['ph_final'].values
ph_mean = np.nanmean(ph_arr)
ph_c  = ph_arr - ph_mean;  ph_c[np.isnan(ph_c)]  = 0.0
ph_c2 = ph_c ** 2
ph_c3 = ph_c ** 3
ph_basis = np.column_stack([ph_c, ph_c2, ph_c3])

# Soil coverage report
print('Final covariate coverage:')
for c in ['clay_pct','som_pct','bulk_density','soil_moisture',
          'temp_seasonality','precip_seasonality','log_mine_km','shannon']:
    if c in base.columns:
        n = base[c].notna().sum()
        print(f'  {c}: {n:,}/{len(base):,}')

def impute_median(arr):
    arr = arr.copy()
    mask = np.isnan(arr)
    if mask.any():
        arr[mask] = np.nanmedian(arr)
    return arr

lith_cols   = [c for c in base.columns if c.startswith('lith_')]
region_cols = [c for c in base.columns if c.startswith('region_')]
phy_cols    = [c for c in base.columns if c in top8_phyla]

Z_blocks = {
    'L1': ph_basis,
    'L2': np.column_stack([
        base['clay_pct'].values.astype(np.float64),    # OLM+SoilGrids filled
        base['som_pct'].values.astype(np.float64),
        base['bulk_density'].values.astype(np.float64),
        *[base[c].fillna(0).values.astype(np.float64) for c in lith_cols],
    ]),
    'L3': base['log_mine_km'].values.reshape(-1, 1).astype(np.float64),
    'L4': np.column_stack([
        impute_median(base['era5_mean_2m_air_temp_k'].values),
        impute_median(base['era5_total_precipitation_mm'].values),
        impute_median(base['temp_seasonality'].values),
        impute_median(base['precip_seasonality'].values),
    ]),
    'L5': np.column_stack([
        impute_median(base['shannon'].values),
        *[base[p].fillna(0).values.astype(np.float64) for p in phy_cols],
    ]),
    'L6': np.column_stack([
        impute_median(base['dem_elevation_m'].values),
        base['ndvi'].fillna(0).values.astype(np.float64),
    ]),
}

def build_Z(level_int):
    keys = ['L1','L2','L3','L4','L5','L6'][:level_int]
    if not keys:
        return np.ones((len(base), 1))
    parts = [np.ones((len(base), 1))] + [Z_blocks[k] for k in keys]
    if region_cols:
        parts.append(np.column_stack([base[c].fillna(0).values for c in region_cols]))
    return np.column_stack(parts).astype(np.float64)

print('\nCovariate matrix shapes:')
for lv in range(7):
    Z = build_Z(lv)
    n_finite = np.all(np.isfinite(Z), axis=1).sum()
    print(f'  L{lv}: {Z.shape}, rows fully finite: {n_finite:,}')


Final covariate coverage:
  clay_pct: 4,781/4,884
  som_pct: 3,843/4,884
  bulk_density: 3,843/4,884
  soil_moisture: 3,806/4,884
  temp_seasonality: 4,883/4,884
  precip_seasonality: 4,883/4,884
  log_mine_km: 4,884/4,884
  shannon: 4,879/4,884

Covariate matrix shapes:
  L0: (4884, 1), rows fully finite: 4,884
  L1: (4884, 6), rows fully finite: 4,884
  L2: (4884, 21), rows fully finite: 3,843
  L3: (4884, 22), rows fully finite: 3,843
  L4: (4884, 26), rows fully finite: 3,843
  L5: (4884, 35), rows fully finite: 3,843
  L6: (4884, 37), rows fully finite: 3,843


In [15]:
# Vectorized FWL for all metals in METAL_LIST x causal levels x all KOs.
# Clip non-positive metal values to NaN before log10 (0 = below detection).
# Reports: beta, SE, t, p, n (cell count), beta_per_iqr (IQR-standardised effect).

def fwl_all_kos(X, Y, Z):
    """FWL: beta, SE, t-stat for each column of Y regressed on X after partialling Z."""
    n, p = Z.shape
    coef_x, *_ = np.linalg.lstsq(Z, X, rcond=None)
    Mx = X - Z @ coef_x
    coef_y, *_ = np.linalg.lstsq(Z, Y, rcond=None)
    My = Y - Z @ coef_y
    MxMx = Mx @ Mx
    betas = My.T @ Mx / MxMx
    resid = My - np.outer(Mx, betas)
    dof   = max(n - p - 1, 1)
    sigma2 = (resid ** 2).sum(axis=0) / dof
    se     = np.sqrt(sigma2 / MxMx)
    t_stat = betas / np.where(se > 0, se, np.nan)
    return betas, se, t_stat

cwm_aligned = cwm_wide.reindex(base.index).fillna(0.0).values.astype(np.float64)
ko_ids = cwm_wide.columns.tolist()
combined_idx = combined.set_index('sample_id')
print(f'CWM aligned: {cwm_aligned.shape}')
print(f'Running FWL for {len(METAL_LIST)} elements x 7 causal levels...')

all_results = []
from scipy.stats import t as t_dist

for metal in METAL_LIST:
    metal_arr = pd.to_numeric(combined_idx.reindex(base.index)[metal], errors='coerce').values
    metal_arr = np.where(metal_arr > 0, metal_arr, np.nan)  # 0/negative = below detection
    log_metal = np.log10(metal_arr)
    iqr_log   = np.nanpercentile(log_metal, 75) - np.nanpercentile(log_metal, 25)

    for level in range(7):
        Z = build_Z(level)
        valid = np.isfinite(log_metal) & np.all(np.isfinite(Z), axis=1)
        n_valid = int(valid.sum())
        if n_valid < 30:
            continue

        X  = log_metal[valid]
        Y  = cwm_aligned[valid]
        Zs = Z[valid]

        betas, se, t_stat = fwl_all_kos(X, Y, Zs)
        dof   = max(n_valid - Zs.shape[1] - 1, 1)
        pvals = 2 * t_dist.sf(np.abs(t_stat), df=dof)
        beta_per_iqr = betas * iqr_log

        df_res = pd.DataFrame({
            'metal':        metal,
            'level':        f'L{level}',
            'ko_id':        ko_ids,
            'n':            n_valid,
            'beta':         betas,
            'se':           se,
            't_stat':       t_stat,
            'p':            pvals,
            'beta_per_iqr': beta_per_iqr,
            'iqr_log':      iqr_log,
        })
        all_results.append(df_res)
        n_nom = int((pvals < 0.05).sum())
        if metal in PRIMARY_METALS or level == 1:
            print(f'  {metal} L{level}: n={n_valid:,}, nom p<0.05: {n_nom:,}')

results = pd.concat(all_results, ignore_index=True)
print(f'\nTotal result rows: {len(results):,}')
results.attrs = {}
results.to_parquet(DATA / 'nb02_fwl_results.parquet', index=False)


CWM aligned: (4884, 6557)
Running FWL for 23 elements x 7 causal levels...


  As L0: n=1,143, nom p<0.05: 1,580


  As L1: n=1,143, nom p<0.05: 518


  As L2: n=997, nom p<0.05: 445


  As L3: n=997, nom p<0.05: 272


...

  Zn L2: n=1,033, nom p<0.05: 760


  Zn L3: n=1,033, nom p<0.05: 758


  Zn L4: n=1,033, nom p<0.05: 630
  Zn L5: n=1,033, nom p<0.05: 296


  Zn L6: n=1,033, nom p<0.05: 304
  Zr L1: n=531, nom p<0.05: 690



Total result rows: 1,055,677


In [16]:
from statsmodels.stats.multitest import multipletests

results_fdr = []
for (metal, level), grp in results.groupby(['metal','level']):
    valid_p = grp['p'].notna() & grp['p'].between(0, 1)
    q = np.full(len(grp), np.nan)
    if valid_p.sum() > 0:
        _, q_vals, _, _ = multipletests(grp.loc[valid_p, 'p'], method='fdr_bh')
        q[valid_p.values] = q_vals
    grp = grp.copy()
    grp['q_bh'] = q
    results_fdr.append(grp)

results_fdr = pd.concat(results_fdr, ignore_index=True)
results_fdr.attrs = {}
results_fdr.to_parquet(DATA / 'nb02_fwl_results_fdr.parquet', index=False)
results_fdr.to_csv(DATA / 'nb02_fwl_results_fdr.csv', index=False)

# Summary: L1 FDR hits (primary estimand)
L1_hits = results_fdr[(results_fdr['level']=='L1') & (results_fdr['q_bh'] < 0.05)]
print('FDR-significant KOs at L1 (pH-adjusted), by metal:')
print(L1_hits.groupby('metal')['ko_id'].count().to_string())
print(f'\nTotal L1 FDR hits: {len(L1_hits):,}')

L0_hits   = set(results_fdr[(results_fdr['level']=='L0') & (results_fdr['q_bh']<0.05)][['metal','ko_id']].apply(tuple, axis=1))
L6_hits   = set(results_fdr[(results_fdr['level']=='L6') & (results_fdr['q_bh']<0.05)][['metal','ko_id']].apply(tuple, axis=1))
L1_hits_set = set(L1_hits[['metal','ko_id']].apply(tuple, axis=1))
print(f'\nL0 hits: {len(L0_hits):,}  L1 hits: {len(L1_hits_set):,}  L6 hits: {len(L6_hits):,}')
print(f'L1 intersect L6 (stable): {len(L1_hits_set & L6_hits):,}')

# Top hits with n and IQR effect
print('\nTop 20 L1 FDR hits (sorted by q_bh):')
if len(L1_hits) > 0:
    top20 = L1_hits.sort_values('q_bh').head(20)
    print(top20[['metal','ko_id','n','beta','beta_per_iqr','q_bh']].to_string(index=False))


FDR-significant KOs at L1 (pH-adjusted), by metal:
metal
As     41
Cd     10
Cr     38
La      9
Nb      1
Nd    235
Ni     38
Pb     14
Sr      1
V       6
Yb     91
Zn     76

Total L1 FDR hits: 560

L0 hits: 1,433  L1 hits: 560  L6 hits: 54
L1 intersect L6 (stable): 16

Top 20 L1 FDR hits (sorted by q_bh):
metal  ko_id    n      beta  beta_per_iqr     q_bh
   Zn K09162 1184  0.007006      0.002834 0.002327
   Cd K09162  998  0.008066      0.003216 0.003359
   Pb K18355 1630  0.002272      0.000766 0.006458
   Pb K09162 1630  0.006585      0.002222 0.006458
   Pb K18363 1630  0.002211      0.000746 0.006458
   Pb K06441 1630  0.003379      0.001140 0.006458
   Pb K04108 1630  0.002212      0.000746 0.007008
   As K12092 1143 -0.003382     -0.001723 0.007718
   As K12089 1143 -0.003946     -0.002010 0.007718
   As K12088 1143 -0.003946     -0.002010 0.007718
   As K12087 1143 -0.003382     -0.001723 0.007718
   As K12091 1143 -0.003995     -0.002036 0.007718
   As K12086 1143 -0.00394

In [17]:
# Manhattan-style plot: −log10(q) for each KO at L1, one panel per metal.
# KOs sorted by genomic order (arbitrary alphabetical here as no genomic context).
metals_available = [m for m in PRIMARY_METALS
                    if len(results_fdr[(results_fdr['metal']==m)&(results_fdr['level']=='L1')]) > 0]
n_metals = len(metals_available)
fig, axs = plt.subplots(1, n_metals, figsize=(FIGW['full'], ROW_H * 0.9), sharey=True)
if n_metals == 1:
    axs = [axs]

for ax, metal in zip(axs, metals_available):
    df = results_fdr[(results_fdr['metal']==metal) & (results_fdr['level']=='L1')].copy()
    df = df.sort_values('ko_id').reset_index(drop=True)
    y = -np.log10(df['q_bh'].clip(1e-20))
    colors = np.where(df['q_bh'] < 0.05, METAL_COLORS.get(metal, PALETTE[0]), '#aaaaaa')
    ax.scatter(range(len(df)), y, c=colors, s=1, alpha=0.6, linewidths=0)
    ax.axhline(-np.log10(0.05), color='gray', lw=0.8, ls='--')
    ax.set_title(metal, fontsize=10)
    ax.set_xlabel('KO rank')
    if ax is axs[0]:
        ax.set_ylabel('−log₁₀(q_BH)')
    n_sig = (df['q_bh'] < 0.05).sum()
    ax.annotate(f'n={len(df):,}\n{n_sig} FDR<0.05', xy=(0.98, 0.98),
                xycoords='axes fraction', ha='right', va='top', fontsize=8, color='#808080')

fig.suptitle('Metal × CWM associations — L1 (pH-adjusted)', y=1.02)
save(fig, FIGS / 'fig_nb02_manhattan_L1')

In [18]:
# L0→L6 stability plot for FDR hits at L1.
# Show how beta changes as confounders are added; highlights attenuation/amplification.
if len(L1_hits) == 0:
    print('No L1 FDR hits — skipping stability plot')
else:
    # Take up to 20 hits sorted by q_bh
    top_hits = L1_hits.sort_values('q_bh').head(20)[['metal','ko_id']]
    
    levels = ['L0','L1','L2','L3','L4','L5','L6']
    fig, ax = plt.subplots(figsize=(FIGW['1.5col'], ROW_H))

    for _, row in top_hits.iterrows():
        trace = []
        for lv in levels:
            r = results_fdr[(results_fdr['metal']==row['metal']) &
                            (results_fdr['level']==lv) &
                            (results_fdr['ko_id']==row['ko_id'])]
            trace.append(r['beta'].values[0] if len(r) else np.nan)
        color = METAL_COLORS.get(row['metal'], PALETTE[0])
        ax.plot(range(len(levels)), trace, color=color, lw=0.7, alpha=0.6)

    ax.axhline(0, color='gray', lw=0.8, ls='--')
    ax.set_xticks(range(len(levels)))
    ax.set_xticklabels(levels)
    ax.set_xlabel('Causal level')
    ax.set_ylabel('FWL beta (log₁₀ metal → CWM)')
    ax.set_title('Beta stability L0→L6 for L1 FDR hits', fontsize=10)

    # Legend: one patch per metal with L1 hits
    metals_hit = top_hits['metal'].unique()
    patches = [mpatches.Patch(color=METAL_COLORS.get(m, PALETTE[i]), label=m)
               for i, m in enumerate(metals_hit)]
    ax.legend(handles=patches, fontsize=8, frameon=False)
    fig.suptitle(f'Top {len(top_hits)} L1 FDR hits (up to 20)', y=1.02)
    save(fig, FIGS / 'fig_nb02_beta_stability')

In [19]:
# Scatter: log10(metal) vs CWM for top 6 L1 FDR hits.
if len(L1_hits) == 0:
    print('No L1 FDR hits — skipping scatter plots')
else:
    top6 = L1_hits.sort_values('q_bh').head(6)

    fig, axs = plt.subplots(2, 3, figsize=(FIGW['full'], ROW_H * 2))
    axs_flat = axs.flatten()

    for i, (_, row) in enumerate(top6.iterrows()):
        metal = row['metal']
        ko    = row['ko_id']
        ax    = axs_flat[i]

        metal_vals = combined.set_index('sample_id').reindex(base.index)[metal].values
        log_metal  = np.log10(pd.to_numeric(metal_vals, errors='coerce').clip(1e-6))
        cwm_vals   = cwm_wide[ko].reindex(base.index).fillna(0).values if ko in cwm_wide.columns else np.zeros(len(base))
        valid = np.isfinite(log_metal)

        color = METAL_COLORS.get(metal, PALETTE[0])
        ax.scatter(log_metal[valid], cwm_vals[valid], s=4, alpha=0.4, color=color, linewidths=0)
        # Regression line (L0 for display)
        if valid.sum() > 5:
            z = np.polyfit(log_metal[valid], cwm_vals[valid], 1)
            xr = np.linspace(log_metal[valid].min(), log_metal[valid].max(), 50)
            ax.plot(xr, np.polyval(z, xr), color='k', lw=0.8)

        ax.set_xlabel(f'log₁₀({metal} ppm)')
        ax.set_ylabel(f'CWM ({ko})')
        ax.set_title(f'{metal} × {ko}', fontsize=9)
        q_str = f"q={row['q_bh']:.2e}"
        ax.annotate(q_str, xy=(0.98,0.02), xycoords='axes fraction',
                    ha='right', va='bottom', fontsize=8, color='#808080')

    # Hide unused panels if fewer than 6 hits
    for j in range(len(top6), 6):
        axs_flat[j].set_visible(False)

    fig.suptitle('Top L1 FDR hits: log₁₀(metal) vs CWM', y=1.02)
    save(fig, FIGS / 'fig_nb02_scatter_top_hits')

In [20]:
# Reverse direction: predict log10(metal) from CWM using feature-selected Ridge.
# Feature selection: per-metal L1 FDR KOs (determined system: 5–250 features vs ~1,500 samples).
# Two CV schemes:
#   spatial_cv: K-means spatial blocks (5 folds) — tests cross-region extrapolation
#   random_cv:  random 5-fold CV — tests within-region interpolation (inflated by spatial autocorr)
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold

_fdr_full = pd.read_parquet(DATA / 'nb02_fwl_results_fdr.parquet')
_L1_hits  = _fdr_full[(_fdr_full['level'] == 'L1') & (_fdr_full['q_bh'] < 0.05)]

ALPHAS = [0.1, 1, 10, 100, 1000, 10_000, 100_000]
reverse_results = []

def ridge_cv_r2(X_feat, y, fold_iter):
    scaler = StandardScaler()
    r2_scores, best_alphas = [], []
    for tr, te in fold_iter:
        if len(tr) < 20 or len(te) < 3:
            continue
        X_tr = scaler.fit_transform(X_feat[tr])
        X_te = scaler.transform(X_feat[te])
        rcv  = RidgeCV(alphas=ALPHAS, scoring='r2', cv=3)
        rcv.fit(X_tr, y[tr])
        best_alphas.append(rcv.alpha_)
        y_pred = rcv.predict(X_te)
        ss_res = ((y[te] - y_pred) ** 2).sum()
        ss_tot = ((y[te] - y[te].mean()) ** 2).sum()
        r2_scores.append(1 - ss_res / ss_tot if ss_tot > 0 else np.nan)
    mean_r2      = float(np.nanmean(r2_scores)) if r2_scores else np.nan
    median_alpha = float(np.median(best_alphas)) if best_alphas else np.nan
    return mean_r2, median_alpha

for metal in METAL_LIST:
    hits_kos = set(_L1_hits[_L1_hits['metal'] == metal]['ko_id'].values)
    if len(hits_kos) < 5:
        print(f'{metal}: {len(hits_kos)} L1 FDR hits — skipped (< 5 features)')
        continue

    hit_idx = [i for i, k in enumerate(ko_ids) if k in hits_kos]
    if len(hit_idx) < 5:
        continue

    metal_arr = pd.to_numeric(
        combined.set_index('sample_id').reindex(base.index)[metal], errors='coerce'
    ).values
    metal_arr = np.where(metal_arr > 0, metal_arr, np.nan)
    log_metal = np.log10(metal_arr)
    valid     = np.isfinite(log_metal) & np.all(np.isfinite(cwm_aligned[:, hit_idx]), axis=1)
    n_valid   = int(valid.sum())
    if n_valid < 30:
        continue

    X_feat = cwm_aligned[valid][:, hit_idx]
    y      = log_metal[valid]
    coords = base[['lat', 'lon']].values[valid]

    # Spatial block CV
    n_blocks = min(5, n_valid // 20)
    km     = KMeans(n_clusters=n_blocks, random_state=42, n_init=10)
    blocks = km.fit_predict(coords)
    sp_folds = [(np.where(blocks != b)[0], np.where(blocks == b)[0]) for b in range(n_blocks)]
    r2_sp, alpha_sp = ridge_cv_r2(X_feat, y, sp_folds)

    # Random 5-fold CV
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    r2_rnd, alpha_rnd = ridge_cv_r2(X_feat, y, list(kf.split(X_feat)))

    n_feat = len(hit_idx)
    print(f'{metal}: n={n_valid:,}, p={n_feat}, spatial R²={r2_sp:.4f}, random R²={r2_rnd:.4f}')
    reverse_results.append({'metal': metal, 'n': n_valid, 'n_features': n_feat,
                            'r2_spatial_cv': r2_sp, 'r2_random_cv': r2_rnd,
                            'alpha_spatial': alpha_sp, 'alpha_random': alpha_rnd})

reverse_df = pd.DataFrame(reverse_results)
reverse_df.to_csv(DATA / 'nb02_reverse_ridge_r2.csv', index=False)
print(reverse_df[['metal','n','n_features','r2_spatial_cv','r2_random_cv']].to_string(index=False))


As: n=1,143, p=41, spatial R²=-3.6585, random R²=0.0359
B: 0 L1 FDR hits — skipped (< 5 features)
Ba: 0 L1 FDR hits — skipped (< 5 features)
Be: 0 L1 FDR hits — skipped (< 5 features)
Cd: n=998, p=10, spatial R²=-0.2672, random R²=0.0015
Ce: 0 L1 FDR hits — skipped (< 5 features)
Co: 0 L1 FDR hits — skipped (< 5 features)


Cr: n=1,693, p=38, spatial R²=-1.1313, random R²=-0.0036
Cu: 0 L1 FDR hits — skipped (< 5 features)
Ga: 0 L1 FDR hits — skipped (< 5 features)
La: n=295, p=9, spatial R²=-20.0283, random R²=-0.0030
Mo: 0 L1 FDR hits — skipped (< 5 features)
Nb: 1 L1 FDR hits — skipped (< 5 features)


Nd: n=89, p=235, spatial R²=-0.6481, random R²=-0.8303


Ni: n=1,645, p=38, spatial R²=-0.0544, random R²=-0.0042
Pb: n=1,630, p=14, spatial R²=-0.1617, random R²=0.0256
Sc: 0 L1 FDR hits — skipped (< 5 features)
Sr: 1 L1 FDR hits — skipped (< 5 features)


V: n=526, p=6, spatial R²=-0.4615, random R²=0.0364
Y: 0 L1 FDR hits — skipped (< 5 features)


Yb: n=433, p=91, spatial R²=-0.0011, random R²=0.0623


Zn: n=1,184, p=76, spatial R²=-1.5015, random R²=0.0306
Zr: 0 L1 FDR hits — skipped (< 5 features)
metal    n  n_features  r2_spatial_cv  r2_random_cv
   As 1143          41      -3.658455      0.035898
   Cd  998          10      -0.267248      0.001456
   Cr 1693          38      -1.131275     -0.003613
   La  295           9     -20.028328     -0.003015
   Nd   89         235      -0.648118     -0.830262
   Ni 1645          38      -0.054365     -0.004157
   Pb 1630          14      -0.161685      0.025646
    V  526           6      -0.461543      0.036400
   Yb  433          91      -0.001100      0.062263
   Zn 1184          76      -1.501526      0.030567


In [21]:
# Positive control: predict soil pH from CWM using RidgeCV.
# Feature selection: top-200 KOs by abs(Pearson r) with pH (globally — this is a positive control,
# optimistic bias is acceptable; point is to show the method has power when signal exists).
# Same two CV schemes as metals for direct comparison.
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold

ph_arr = base['ph_final'].values
ph_valid_mask = np.isfinite(ph_arr)

# Select top-200 KOs by |correlation with pH| (globally)
cwm_ph = cwm_aligned[ph_valid_mask]
ph_sub  = ph_arr[ph_valid_mask]
corrs   = np.array([
    np.corrcoef(cwm_ph[:, j], ph_sub)[0, 1]
    if np.std(cwm_ph[:, j]) > 0 else 0.0
    for j in range(cwm_ph.shape[1])
])
top200_idx = np.argsort(np.abs(corrs))[::-1][:200]

valid  = np.isfinite(ph_arr) & np.all(np.isfinite(cwm_aligned[:, top200_idx]), axis=1)
n_valid = int(valid.sum())
print(f'pH positive control: n={n_valid:,}, features=200 (top |r| with pH)')

X_feat = cwm_aligned[valid][:, top200_idx]
y      = ph_arr[valid]
coords = base[['lat', 'lon']].values[valid]

ALPHAS = [0.1, 1, 10, 100, 1000, 10_000, 100_000]

def ridge_cv_r2(X_feat, y, fold_iter):
    scaler = StandardScaler()
    r2_scores, best_alphas = [], []
    for tr, te in fold_iter:
        if len(tr) < 20 or len(te) < 3:
            continue
        X_tr = scaler.fit_transform(X_feat[tr])
        X_te = scaler.transform(X_feat[te])
        rcv  = RidgeCV(alphas=ALPHAS, scoring='r2', cv=3)
        rcv.fit(X_tr, y[tr])
        best_alphas.append(rcv.alpha_)
        y_pred = rcv.predict(X_te)
        ss_res = ((y[te] - y_pred) ** 2).sum()
        ss_tot = ((y[te] - y[te].mean()) ** 2).sum()
        r2_scores.append(1 - ss_res / ss_tot if ss_tot > 0 else np.nan)
    return float(np.nanmean(r2_scores)) if r2_scores else np.nan, float(np.median(best_alphas)) if best_alphas else np.nan

# Spatial block CV
n_blocks = min(5, n_valid // 20)
km = KMeans(n_clusters=n_blocks, random_state=42, n_init=10)
blocks = km.fit_predict(coords)
sp_folds = [(np.where(blocks != b)[0], np.where(blocks == b)[0]) for b in range(n_blocks)]
r2_sp, alpha_sp = ridge_cv_r2(X_feat, y, sp_folds)

# Random 5-fold CV
kf = KFold(n_splits=5, shuffle=True, random_state=42)
r2_rnd, alpha_rnd = ridge_cv_r2(X_feat, y, list(kf.split(X_feat)))

print(f'pH: n={n_valid:,}, features=200, spatial R²={r2_sp:.4f}, random R²={r2_rnd:.4f}')
ph_ctrl_row = {'metal': 'pH (ctrl)', 'n': n_valid, 'n_features': 200,
               'r2_spatial_cv': r2_sp, 'r2_random_cv': r2_rnd,
               'alpha_spatial': alpha_sp, 'alpha_random': alpha_rnd}


pH positive control: n=4,844, features=200 (top |r| with pH)


pH: n=4,844, features=200, spatial R²=0.1312, random R²=0.2598


In [22]:
# Figure: paired spatial and random CV R² for metals + pH positive control.
# Spatial CV tests cross-region extrapolation; random CV tests within-region interpolation.
all_rows = list(reverse_results) + [ph_ctrl_row]
plot_df  = pd.DataFrame(all_rows).sort_values('r2_random_cv', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(FIGW['2col'], ROW_H))
x  = np.arange(len(plot_df))
w  = 0.35

bar_colors = [('#2CA02C' if m == 'pH (ctrl)' else METAL_COLORS.get(m, PALETTE[0]))
              for m in plot_df['metal']]

bars_rnd = ax.bar(x - w/2, plot_df['r2_random_cv'].clip(-1.5), w,
                  color=bar_colors, edgecolor='k', linewidth=0.5, label='Random 5-fold CV')
bars_sp  = ax.bar(x + w/2, plot_df['r2_spatial_cv'].clip(-1.5), w,
                  color=bar_colors, edgecolor='k', linewidth=0.5, alpha=0.35, label='Spatial block CV')

ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set_xticks(x)
ax.set_xticklabels(plot_df['metal'], rotation=45, ha='right')
ax.set_xlabel('Target variable')
ax.set_ylabel('CV R²')
ax.set_title('Reverse Ridge: CWM → target  (solid=random CV, faded=spatial block CV)', fontsize=10)
ax.legend(fontsize=8, loc='upper right')
grid_h(ax)

for xi, (_, rw) in enumerate(plot_df.iterrows()):
    ypos = max(rw['r2_random_cv'], -1.5) + 0.03
    ax.annotate(f'p={int(rw["n_features"])}', (xi - w/2, ypos),
                ha='center', fontsize=7, color='#808080')

save(fig, FIGS / 'fig_nb02_reverse_r2')


In [23]:
# FWL forward analysis: pH as predictor → CWM.
# Same vectorized FWL as metals, but exposure X = ph_final (raw pH, already log-transformed).
# Z controls: intercept + lithology dummies + climate (MAT, MAP, temp/precip seasonality).
# Deliberately excludes soil properties (clay/SOC/CEC) — these partially mediate pH's effect.
# BH-FDR within the 6,557 KOs.
from statsmodels.stats.multitest import multipletests

ph_X = base['ph_final'].values.astype(np.float64)

# Z: intercept + lat + lon (linear, to remove broad geographic gradient).
# Do NOT control for climate or lithology — both mediate pH's effect on communities
# (climate → weathering → pH; lithology → buffering → pH), so including them
# over-controls and removes the pH signal from FWL residuals.
lat_c = (base['lat'].values  - base['lat'].mean())  / base['lat'].std()
lon_c = (base['lon'].values  - base['lon'].mean())  / base['lon'].std()
Z_ph = np.column_stack([
    np.ones(len(base)),
    lat_c,
    lon_c,
])

valid_ph = np.isfinite(ph_X) & np.all(np.isfinite(Z_ph), axis=1)
n_ph = int(valid_ph.sum())
print(f'pH FWL: n={n_ph:,} valid samples, p={Z_ph.shape[1]} covariates')

betas_ph, se_ph, t_ph = fwl_all_kos(ph_X[valid_ph], cwm_aligned[valid_ph], Z_ph[valid_ph])
pvals_ph = 2 * (1 - norm.cdf(np.abs(t_ph)))
# NaN t_stats (from zero-variance KOs) break multipletests — mask before BH
q_ph = np.full_like(pvals_ph, np.nan)
valid_p = np.isfinite(pvals_ph)
if valid_p.sum() > 0:
    _, q_ph[valid_p], _, _ = multipletests(pvals_ph[valid_p], method='fdr_bh')

iqr_ph = np.nanpercentile(ph_X[valid_ph], 75) - np.nanpercentile(ph_X[valid_ph], 25)
beta_per_iqr_ph = betas_ph * iqr_ph

fwl_ph_df = pd.DataFrame({
    'ko_id': ko_ids, 'n': n_ph,
    'beta': betas_ph, 'se': se_ph, 't_stat': t_ph,
    'p': pvals_ph, 'q_bh': q_ph,
    'beta_per_iqr': beta_per_iqr_ph, 'iqr': iqr_ph,
})
fwl_ph_df.attrs = {}
fwl_ph_df.to_parquet(DATA / 'nb02_fwl_ph_results.parquet', index=False)

n_ph_hits = (q_ph < 0.05).sum()
print(f'pH FDR hits (q<0.05): {n_ph_hits:,} / {len(ko_ids):,} KOs')
print('Top 10 pH FDR hits:')
top_ph = fwl_ph_df[fwl_ph_df['q_bh'] < 0.05].sort_values('q_bh').head(10)
print(top_ph[['ko_id','n','beta','beta_per_iqr','q_bh']].to_string(index=False))


pH FWL: n=4,844 valid samples, p=3 covariates


pH FDR hits (q<0.05): 4,219 / 6,557 KOs
Top 10 pH FDR hits:
 ko_id    n      beta  beta_per_iqr  q_bh
K18011 4844  0.009413      0.016943   0.0
K18313 4844  0.012446      0.022404   0.0
K18306 4844 -0.004582     -0.008248   0.0
K18305 4844  0.007237      0.013026   0.0
K18294 4844  0.009811      0.017659   0.0
K06320 4844  0.013487      0.024277   0.0
K06217 4844  0.025021      0.045037   0.0
K06208 4844  0.015016      0.027030   0.0
K18012 4844  0.006818      0.012273   0.0
K06384 4844  0.016915      0.030447   0.0


In [24]:
# Extended facultative KO test: compare pH hits, per-metal hits, and background.
# Lower mean_prev within genus = more facultative = more gene-gain-consistent.
from scipy.stats import mannwhitneyu

ko_prev = pd.read_parquet(DATA / 'nb01_ko_prevalence.parquet')

ko_stats_all = (
    ko_prev[ko_prev['prevalence'] > 0]
    .groupby('ko_id')
    .agg(mean_prev=('prevalence', 'mean'),
         frac_facultative=('prevalence', lambda x: ((x > 0.2) & (x < 0.8)).mean()),
         n_genera=('prevalence', 'count'))
    .reset_index()
)
ko_stats_cwm2 = ko_stats_all[ko_stats_all['ko_id'].isin(set(ko_ids))].copy()

# Define groups
fdr_data  = pd.read_parquet(DATA / 'nb02_fwl_results_fdr.parquet')
L1_hits   = fdr_data[(fdr_data['level'] == 'L1') & (fdr_data['q_bh'] < 0.05)]
ph_fwl    = pd.read_parquet(DATA / 'nb02_fwl_ph_results.parquet')
ph_hit_kos = set(ph_fwl[ph_fwl['q_bh'] < 0.05]['ko_id'])
metal_hit_kos = set(L1_hits['ko_id'])

bkg_kos = set(ko_ids) - ph_hit_kos - metal_hit_kos

groups = {
    'Background': bkg_kos,
    'Metal hits (any)': metal_hit_kos,
    'pH hits': ph_hit_kos,
}

summary_rows = []
for grp, kos in groups.items():
    vals = ko_stats_cwm2[ko_stats_cwm2['ko_id'].isin(kos)]['mean_prev']
    summary_rows.append({'group': grp, 'n': len(vals),
                         'median_mean_prev': vals.median(),
                         'mean_mean_prev': vals.mean()})
    print(f'{grp:22s}: n={len(vals):,}  median mean_prev={vals.median():.3f}')

# Mann-Whitney vs background for each group
bkg_vals = ko_stats_cwm2[ko_stats_cwm2['ko_id'].isin(bkg_kos)]['mean_prev']
for grp, kos in [('Metal hits', metal_hit_kos), ('pH hits', ph_hit_kos)]:
    vals = ko_stats_cwm2[ko_stats_cwm2['ko_id'].isin(kos)]['mean_prev']
    _, p = mannwhitneyu(vals, bkg_vals, alternative='two-sided')
    print(f'{grp} vs background: p={p:.3g}')

summary_df = pd.DataFrame(summary_rows)

# Per-metal breakdown (add pH row)
facult_df_ext = pd.read_csv(DATA / 'nb02_facult_ko.csv')
ph_mp = ko_stats_cwm2[ko_stats_cwm2['ko_id'].isin(ph_hit_kos)]['mean_prev']
ph_ff = ko_stats_cwm2[ko_stats_cwm2['ko_id'].isin(ph_hit_kos)]['frac_facultative']
facult_df_ext = pd.concat([
    pd.DataFrame([{'metal': 'pH', 'n_kos': len(ph_mp),
                   'median_mean_prev': ph_mp.median(),
                   'median_frac_facult': ph_ff.median()}]),
    facult_df_ext
], ignore_index=True)
facult_df_ext.to_csv(DATA / 'nb02_facult_ko_extended.csv', index=False)
print('\nExtended per-predictor summary:')
print(facult_df_ext[['metal','n_kos','median_mean_prev']].sort_values('median_mean_prev').to_string(index=False))


Background            : n=2,135  median mean_prev=0.602
Metal hits (any)      : n=427  median mean_prev=0.601
pH hits               : n=4,219  median mean_prev=0.735
Metal hits vs background: p=0.135
pH hits vs background: p=2.79e-76

Extended per-predictor summary:
metal  n_kos  median_mean_prev
   Nb      1          0.367063
   Yb    124          0.381076
   Ni     41          0.583333
   Cr     41          0.583333
   Zn     76          0.583333
   Pb     14          0.606692
   As     48          0.611521
   Nd    250          0.640919
   Co      1          0.648186
   La     12          0.696393
   Sr      1          0.701762
   pH   4219          0.735094
    V      7          0.794419
   Cd     25          0.806372
   Cu      6          0.854167


In [25]:
# Figure: facultativeness across predictors — pH hits, per-metal hits, background.
import matplotlib.patches as mpatches
from scipy.stats import mannwhitneyu

ko_stats_plot = ko_stats_cwm2.copy()
facult_ext = pd.read_csv(DATA / 'nb02_facult_ko_extended.csv')

bkg_mp   = ko_stats_plot[ko_stats_plot['ko_id'].isin(bkg_kos)]['mean_prev']
metal_mp = ko_stats_plot[ko_stats_plot['ko_id'].isin(metal_hit_kos)]['mean_prev']
ph_mp2   = ko_stats_plot[ko_stats_plot['ko_id'].isin(ph_hit_kos)]['mean_prev']

fig, axes = plt.subplots(1, 2, figsize=(FIGW['2col'], ROW_H))

# Panel A: three-group violin
ax = axes[0]
grp_data   = [('Background', bkg_mp,   PALETTE[3]),
              ('Metal hits', metal_mp, PALETTE[0]),
              ('pH hits',    ph_mp2,   PALETTE[2])]
for i, (label, vals, color) in enumerate(grp_data):
    parts = ax.violinplot(vals.dropna(), positions=[i], widths=0.6,
                          showmedians=True, showextrema=False)
    for pc in parts['bodies']:
        pc.set_facecolor(color); pc.set_alpha(0.7)
        pc.set_edgecolor('k'); pc.set_linewidth(0.5)
    parts['cmedians'].set_color('k'); parts['cmedians'].set_linewidth(1.5)
    ax.annotate(f'n={len(vals):,}\n{vals.median():.3f}',
                (i, -0.08), ha='center', fontsize=7, color='#808080')

# significance brackets
def sig_bracket(ax, x1, x2, y, p):
    s = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'ns'))
    ax.plot([x1, x2], [y, y], 'k-', lw=0.8)
    ax.text((x1+x2)/2, y+0.02, s, ha='center', fontsize=9)

_, p_met = mannwhitneyu(metal_mp, bkg_mp, alternative='two-sided')
_, p_ph  = mannwhitneyu(ph_mp2,   bkg_mp, alternative='two-sided')
sig_bracket(ax, 0, 1, 1.08, p_met)
sig_bracket(ax, 0, 2, 1.16, p_ph)

ax.set_xticks([0,1,2])
ax.set_xticklabels(['Background', 'Metal\nhits', 'pH\nhits'])
ax.set_ylabel('Mean KO prevalence within genus')
ax.set_title('KO facultativeness by predictor type\n(lower = more gene-gain-consistent)', fontsize=10)
ax.set_ylim(-0.15, 1.3)
grid_h(ax)

# Panel B: per-predictor dot plot
ax = axes[1]
facult_sorted = facult_ext.sort_values('median_mean_prev').reset_index(drop=True)
bkg_med = bkg_mp.median()
colors2 = ['#2CA02C' if m == 'pH' else METAL_COLORS.get(m, PALETTE[0])
           for m in facult_sorted['metal']]
ax.scatter(facult_sorted['median_mean_prev'], range(len(facult_sorted)),
           c=colors2, s=40, zorder=3, edgecolors='k', linewidths=0.5)
ax.axvline(bkg_med, color='gray', lw=0.8, ls='--', label=f'Background ({bkg_med:.3f})')
ax.set_yticks(range(len(facult_sorted)))
ax.set_yticklabels(facult_sorted['metal'])
ax.set_xlabel('Median mean KO prevalence within genus')
ax.set_title('Per-predictor KO facultativeness', fontsize=10)
ax.legend(fontsize=7)
grid_h(ax)

fig.suptitle('Gene gain vs. turnover: facultative KO analysis', y=1.02)
save(fig, FIGS / 'fig_nb02_facult_extended')


In [26]:
# GlobDB MRG supplement: FWL for 4 canonical MRGs absent from ke_pangenome
# merA K00221, merB K07444, zntA/cadA K01534, czcA K16786
# Uses GlobDB (SPIRE+MGnify MAG) genus-level prevalences for genera
# that overlap with ke_pangenome (~1,387 genera, ~17% of MA genera).

MRG_KOS   = ['K00221', 'K07444', 'K01534', 'K16786']
MRG_NAMES = {'K00221': 'merA', 'K07444': 'merB', 'K01534': 'zntA/cadA', 'K16786': 'czcA'}

mrg_prev = pd.read_parquet(DATA / 'nb02_globdb_mrg_prevalence.parquet')
print(f'GlobDB MRG prevalences: {len(mrg_prev):,} genus×KO pairs')

mrg_wide = mrg_prev.pivot_table(index='genus_lower', columns='ko_id', values='prevalence', fill_value=0.0)

# Compute genus RA per sample then CWM
total_per_sample = genus_counts.groupby('sample_id')['genus_count'].sum().rename('total')
gc2 = genus_counts.join(total_per_sample, on='sample_id')
gc2['genus_ra'] = gc2['genus_count'] / gc2['total']
gc2 = gc2[gc2['genus_lower'].isin(mrg_wide.index)]

ra_wide = gc2.pivot_table(index='sample_id', columns='genus_lower', values='genus_ra', fill_value=0.0)
shared_genera = ra_wide.columns.intersection(mrg_wide.index)
ra_mat   = ra_wide[shared_genera].values.astype('float64')
prev_mat = mrg_wide.loc[shared_genera, MRG_KOS].values.astype('float64')

cwm_mrg = pd.DataFrame(ra_mat @ prev_mat, index=ra_wide.index, columns=MRG_KOS)
print(f'MRG CWM: {cwm_mrg.shape}')
for ko in MRG_KOS:
    nonzero = (cwm_mrg[ko] > 0).sum()
    print(f'  {ko} ({MRG_NAMES[ko]}): {nonzero:,} samples with CWM>0, mean={cwm_mrg[ko].mean():.4f}')

cwm_mrg_aligned = cwm_mrg.reindex(base.index).fillna(0.0).values.astype('float64')

# FWL across all metals x levels
from scipy.stats import t as t_dist
mrg_results = []
for metal in METAL_LIST:
    metal_arr = pd.to_numeric(combined_idx.reindex(base.index)[metal], errors='coerce').values
    metal_arr = np.where(metal_arr > 0, metal_arr, np.nan)
    log_metal = np.log10(metal_arr)
    iqr_log   = np.nanpercentile(log_metal, 75) - np.nanpercentile(log_metal, 25)
    for level in range(7):
        Z = build_Z(level)
        valid = np.isfinite(log_metal) & np.all(np.isfinite(Z), axis=1)
        n_valid = int(valid.sum())
        if n_valid < 30:
            continue
        betas, se, t_stat = fwl_all_kos(log_metal[valid], cwm_mrg_aligned[valid], Z[valid])
        dof   = max(n_valid - Z.shape[1] - 1, 1)
        pvals = 2 * t_dist.sf(np.abs(t_stat), df=dof)
        for i, ko in enumerate(MRG_KOS):
            mrg_results.append({'metal': metal, 'level': f'L{level}', 'ko_id': ko,
                                 'n': n_valid, 'beta': betas[i], 'se': se[i],
                                 't_stat': t_stat[i], 'p': pvals[i],
                                 'beta_per_iqr': betas[i] * iqr_log})

mrg_df = pd.DataFrame(mrg_results)

from statsmodels.stats.multitest import multipletests
mrg_L1 = mrg_df[mrg_df['level'] == 'L1'].copy()
valid_p = np.isfinite(mrg_L1['p'].values)
q = np.full(len(mrg_L1), np.nan)
if valid_p.sum() > 0:
    _, q[valid_p], _, _ = multipletests(mrg_L1['p'].values[valid_p], method='fdr_bh')
mrg_L1['q_bh'] = q

mrg_hits = mrg_L1[mrg_L1['q_bh'] < 0.05]
print(f'\nGlobDB MRG FDR hits at L1 (BH q<0.05): {len(mrg_hits)}')
if len(mrg_hits):
    print(mrg_hits[['metal','ko_id','n','beta_per_iqr','q_bh']].to_string(index=False))
else:
    print('  No FDR hits')

mrg_df.attrs = {}
mrg_df.to_parquet(DATA / 'nb02_globdb_mrg_fwl.parquet', index=False)
mrg_L1.attrs = {}
mrg_L1.to_parquet(DATA / 'nb02_globdb_mrg_fwl_L1.parquet', index=False)
print('Saved: nb02_globdb_mrg_fwl.parquet, nb02_globdb_mrg_fwl_L1.parquet')


GlobDB MRG prevalences: 895 genus×KO pairs
MRG CWM: (4768, 4)
  K00221 (merA): 4,203 samples with CWM>0, mean=0.0138
  K07444 (merB): 4,468 samples with CWM>0, mean=0.0788
  K01534 (zntA/cadA): 4,744 samples with CWM>0, mean=0.1404
  K16786 (czcA): 4,441 samples with CWM>0, mean=0.0388



GlobDB MRG FDR hits at L1 (BH q<0.05): 1
metal  ko_id   n  beta_per_iqr     q_bh
   Cd K01534 998      0.024447 0.005585
Saved: nb02_globdb_mrg_fwl.parquet, nb02_globdb_mrg_fwl_L1.parquet


In [27]:
# Print a compact summary for REPORT.md
n_L1  = len(L1_hits)
n_L6  = len(L6_hits)
n_stable = len(L1_hits_set & L6_hits)

print('=== NB02 Summary ===')
print(f'Samples with ≥1 primary metal measurement: {combined["region"].notna().sum():,}')
print(f'  USA (USGS): {(combined["region"]=="USA").sum():,}')
print(f'  EUR (GEMAS): {(combined["region"]=="EUR").sum():,}')
print(f'  AUS (NGSA):  {(combined["region"]=="AUS").sum():,}')
print()
print(f'FWL forward (L0–L6):')
print(f'  L1 FDR hits:  {n_L1} (primary estimand)')
print(f'  L6 FDR hits:  {n_L6}')
print(f'  L1 ∩ L6 (stable through full confound control): {n_stable}')
print()
print('L1 hits by metal:')
print(L1_hits.groupby('metal')['ko_id'].count().to_string())
print()
print('Reverse (CWM → metal, Ridge + spatial block CV):')
print(reverse_df[['metal','n','r2_spatial_cv']].to_string(index=False))

=== NB02 Summary ===
Samples with ≥1 primary metal measurement: 1,702
  USA (USGS): 545
  EUR (GEMAS): 921
  AUS (NGSA):  236

FWL forward (L0–L6):
  L1 FDR hits:  560 (primary estimand)
  L6 FDR hits:  54
  L1 ∩ L6 (stable through full confound control): 16

L1 hits by metal:
metal
As     41
Cd     10
Cr     38
La      9
Nb      1
Nd    235
Ni     38
Pb     14
Sr      1
V       6
Yb     91
Zn     76

Reverse (CWM → metal, Ridge + spatial block CV):
metal    n  r2_spatial_cv
   As 1143      -3.658455
   Cd  998      -0.267248
   Cr 1693      -1.131275
   La  295     -20.028328
   Nd   89      -0.648118
   Ni 1645      -0.054365
   Pb 1630      -0.161685
    V  526      -0.461543
   Yb  433      -0.001100
   Zn 1184      -1.501526



## Findings

*(Populate after execution — see summary cell output above)*

### Forward analysis (metal → CWM)

**L1 FDR hits** (pH-adjusted, primary estimand): see `nb02_fwl_results_fdr.csv`.  
**L6 FDR hits** (full confound control): see above.  
**Stable hits** (L1 ∩ L6): KOs whose effect persists through all 6 covariate levels — primary candidates for NB03 functional interpretation.

### Reverse analysis (CWM → metal)

Ridge regression with spatial block CV R² reported in `nb02_reverse_ridge_r2.csv`. Metals with R² > 0.1 indicate that community gene content carries predictive information about metal exposure.

### Data quality notes

- Metal measurements spatially matched ≤ 50 km radius; sample sizes per metal vary by region coverage.
- Soil properties (clay, SOC, bulk density): OLM pre-joined in sample_metadata (primary), SoilGrids spatial join fallback for samples outside OLM coverage. Near-global coverage after fill.
- Soil moisture: OLM only; imputed to column median where missing.
- pH composite source: measured > OLM (×10 corrected). `ph_is_modelled` included as binary covariate in all models.
- GLiM lithology assigned by nearest grid cell (global coverage).
- Mine distance from mindat.csv (157K localities, global coverage).
